In [3]:
!uv pip install  langchain-google-genai

Using Python 3.10.20 environment at: D:\Ai engineering course\.venv
Checked 1 package in 15ms


In [4]:
import logging
import os

In [8]:
"""Step 1 of the PDF RAG: read a PDF, cut it into chunks, store them in ChromaDB."""

import chromadb
from pypdf import PdfReader

PDF_PATH = r"D:\Ai engineering course\Day_33\Exploring_Dog_and_Cat_Management_Practices_in_Mult.pdf"
DB_PATH = r"D:\Ai engineering course\Day_33\chroma_db"   # folder on disk, so the index survives restarts
COLLECTION = "pdf"
CHUNK_SIZE = 1000
OVERLAP = 200


def read_pdf(path):
    """Return one string per page, page numbers kept alongside."""
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append((i, text))
    return pages


def chunk(text, size=CHUNK_SIZE, overlap=OVERLAP):
    """Slide a window over the text; the overlap stops sentences being cut in half."""
    chunks = []
    start = 0
    while start < len(text):
        piece = text[start:start + size].strip()
        if piece:
            chunks.append(piece)
        start += size - overlap
    return chunks


def main():
    pages = read_pdf(PDF_PATH)

    docs, metas, ids = [], [], []
    for page_no, text in pages:
        for j, piece in enumerate(chunk(text)):
            docs.append(piece)
            metas.append({"page": page_no})
            ids.append(f"p{page_no}-c{j}")

    print(f"{len(pages)} pages -> {len(docs)} chunks")

    client = chromadb.PersistentClient(path=DB_PATH)

    if COLLECTION in [c.name for c in client.list_collections()]:
        client.delete_collection(COLLECTION)     # start fresh on a re-run

    collection = client.create_collection(
        COLLECTION,
        metadata={"hnsw:space": "cosine"},       # cosine distance, so 1 - distance = similarity
    )

    # No embeddings passed -> Chroma embeds the text itself (all-MiniLM-L6-v2)
    collection.add(ids=ids, documents=docs, metadatas=metas)

    print(f"stored {collection.count()} chunks in {DB_PATH}")

    # quick sanity check
    hits = collection.query(query_texts=["what is this document about"], n_results=2)
    for doc, meta in zip(hits["documents"][0], hits["metadatas"][0]):
        print(f"\n[page {meta['page']}] {doc[:200]}...")


if __name__ == "__main__":
    main()

19 pages -> 109 chunks
stored 109 chunks in D:\Ai engineering course\Day_33\chroma_db

[page 19] . Vet. Behav. Clin. Appl. Res. 2008, 3, 74–86. [CrossRef]
96. Fox, R.; Gee, N.R. Great Expectations: Changing Social, Spatial and Emotional Understandings of the Companion Animal–Human
Relationship. S...

[page 17] equency of Marking. J. Am. Vet. Med. Assoc. 2001, 219, 1709–1713. [CrossRef] [PubMed]...


In [9]:
"""Step 2: keep the chunks in a variable, and wrap the similarity search as an agent tool."""

import json
import chromadb

# ==========================================
# 1. THE CHUNKS, IN A VARIABLE
# ==========================================
CHUNKS = []      # the text of every chunk
METAS = []       # the page each chunk came from, same order

for page_no, text in read_pdf(PDF_PATH):
    for piece in chunk(text):
        CHUNKS.append(piece)
        METAS.append({"page": page_no})

# The collection step 1 already wrote to disk - just open it, don't rebuild
collection = chromadb.PersistentClient(path=DB_PATH).get_collection(COLLECTION)


# ==========================================
# 2. THE TOOL FUNCTION
# ==========================================
def search_pdf(query, k=3):
    """Search the PDF and return the k most similar chunks as a list of dicts."""
    hits = collection.query(query_texts=[query], n_results=k)

    results = []
    for doc, meta, dist in zip(
        hits["documents"][0], hits["metadatas"][0], hits["distances"][0]
    ):
        results.append({
            "text": doc,
            "page": meta["page"],
            "score": round(1 - dist, 3),   # cosine distance -> similarity
        })
    return results


# ==========================================
# 3. THE SCHEMA THE AGENT READS
# ==========================================
# This is how the model learns the tool exists and when to reach for it.
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_pdf",
        "description": (
            "Search the PDF for passages relevant to a question. "
            "Use this whenever the answer might be in the document."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "What to look for, in plain words.",
                },
                "k": {
                    "type": "integer",
                    "description": "How many chunks to return (default 3).",
                },
            },
            "required": ["query"],
        },
    },
}

# The model can only read strings, so a tool result gets dumped to JSON:
#   args = json.loads(call.function.arguments)
#   output = json.dumps(search_pdf(**args))


if __name__ == "__main__":
    print(f"{len(CHUNKS)} chunks in memory, {collection.count()} in the DB\n")

    for hit in search_pdf("what should I improve", k=2):
        print(f"[page {hit['page']}  score {hit['score']}] {hit['text'][:150]}...\n")

    print(json.dumps(search_pdf("python", k=1))[:200] + "...")

109 chunks in memory, 109 in the DB

[page 7  score 0.175] Female 713 (89.9%) 394 (93.1%)
Age of the respondent (years)
18–25 159 a (20.0%) 53 b (12.5%)
11.4 3 0.01026–40 344 a (43.3%) 192 a (45.4%)
41–55 235 ...

[page 4  score 0.077] s (3.1%) or cats (14.2%), while most of them
owned 1 (56.4%) or 2–5 dogs (40.5%; χ2(2) = 545.8; p < 0.001) and 1 (36.9%) or 2–5 cats
(48.9%; χ2(2) = 2...

[{"text": "riendly.", "page": 9, "score": 0.188}]...
